# Interpretability Pipeline - Task 1 Splicing Prediction

Notebook này triển khai pipeline phân tích interpretability cho mô hình splicing prediction của repo.

Muc tieu:
- Chay duoc end-to-end trong 1 notebook.
- Ho tro 3 phuong phap: Integrated Gradients, Attention Rollout, In-silico Mutagenesis (ISM).
- Bao cao theo donor/acceptor va theo nhom vi tri `all`, `first`, `middle`, `last`.

Luu y:
- Ban se tu patch du lieu va checkpoint vao cac bien PATH o Section 1.
- Dinh nghia mac dinh window = 601, center index = 300.

## 1) Thiet lap duong dan va cau truc thu muc `interpretability/task1_splicing_prediction`

Cell nay tao thu muc con va khai bao PATH trung tam de doc/ghi nhat quan.

In [ ]:
from pathlib import Path
import json
import os

# ===== Centralized PATH config =====
PROJECT_ROOT = Path(r"D:/Bio_sequence_Research_AITALAB")
WORK_ROOT = PROJECT_ROOT / "interpretability" / "task1_splicing_prediction"

DATA_DIR = WORK_ROOT / "data"
CHECKPOINT_DIR = WORK_ROOT / "checkpoints"
OUTPUT_DIR = WORK_ROOT / "outputs"
PLOTS_DIR = WORK_ROOT / "plots"
LOGS_DIR = WORK_ROOT / "logs"
ARTIFACTS_DIR = WORK_ROOT / "artifacts"

for p in [WORK_ROOT, DATA_DIR, CHECKPOINT_DIR, OUTPUT_DIR, PLOTS_DIR, LOGS_DIR, ARTIFACTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ===== User-patched paths (placeholder) =====
INPUT_CSV_PATH = DATA_DIR / "cftr_exon9_sites.csv"
MODEL_CHECKPOINT_PATH = CHECKPOINT_DIR / "best_model.pt"

# Optional: explicit train model file to import architecture if needed
TRAIN_MODEL_PY = PROJECT_ROOT / "train" / "task1_splicing_prediction" / "training" / "model.py"

# ===== Core analysis config =====
WINDOW_LEN = 601
CENTER_INDEX = 300
NT_MODEL_NAME = "InstaDeepAI/nucleotide-transformer-500m-human-ref"
NUM_CLASSES = 3
HIDDEN_DIMS = [512, 256]
DROPOUT = 0.3

CLASS_MAP = {
    0: "negative",
    1: "donor",
    2: "acceptor",
}

VALID_GROUPS = {"all", "first", "middle", "last"}

CONFIG_SNAPSHOT = {
    "input_csv": str(INPUT_CSV_PATH),
    "checkpoint": str(MODEL_CHECKPOINT_PATH),
    "window_len": WINDOW_LEN,
    "center_index": CENTER_INDEX,
    "nt_model": NT_MODEL_NAME,
    "class_map": CLASS_MAP,
}

with open(OUTPUT_DIR / "config_snapshot.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG_SNAPSHOT, f, indent=2)

print("Work root:", WORK_ROOT)
print("Input CSV placeholder:", INPUT_CSV_PATH)
print("Checkpoint placeholder:", MODEL_CHECKPOINT_PATH)

## 2) Cai dat moi truong va import thu vien

Notebook uu tien PyTorch + Captum cho attribution. Neu Captum chua co, cell se thong bao cach cai.

In [ ]:
import random
import warnings
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoTokenizer, AutoModel

try:
    from captum.attr import IntegratedGradients
    CAPTUM_AVAILABLE = True
except Exception:
    CAPTUM_AVAILABLE = False
    IntegratedGradients = None

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("Captum available:", CAPTUM_AVAILABLE)
if not CAPTUM_AVAILABLE:
    print("Install with: pip install captum")

## 3) Nap du lieu cho bai toan splicing prediction

Schema toi thieu de notebook chay:
- `site_id`: id duy nhat cho moi site
- `sequence`: chuoi nucleotide do dai 601 (center la vi tri index 300)
- `label`: nhan so (0/1/2)
- `site_type`: donor/acceptor
- `group_position`: all/first/middle/last

Ban co the bo sung cot genomic nhu `chrom`, `pos`, `strand` neu can bao cao chi tiet.

In [ ]:
REQUIRED_COLUMNS = ["site_id", "sequence", "label", "site_type", "group_position"]


def clean_sequence(seq: str) -> str:
    seq = str(seq).upper().replace("U", "T")
    return "".join([ch if ch in {"A", "C", "G", "T", "N"} else "N" for ch in seq])


def validate_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df.copy()
    df["sequence"] = df["sequence"].map(clean_sequence)

    # Length check and optional auto-fix by center crop/pad
    fixed = []
    for seq in df["sequence"].tolist():
        if len(seq) < WINDOW_LEN:
            left = (WINDOW_LEN - len(seq)) // 2
            right = WINDOW_LEN - len(seq) - left
            seq = ("N" * left) + seq + ("N" * right)
        elif len(seq) > WINDOW_LEN:
            start = (len(seq) - WINDOW_LEN) // 2
            seq = seq[start:start + WINDOW_LEN]
        fixed.append(seq)
    df["sequence"] = fixed

    df["label"] = df["label"].astype(int)
    df["site_type"] = df["site_type"].astype(str).str.lower().str.strip()
    df["group_position"] = df["group_position"].astype(str).str.lower().str.strip()

    valid_site_types = {"donor", "acceptor"}
    bad_types = sorted(set(df["site_type"].unique()) - valid_site_types)
    if bad_types:
        raise ValueError(f"Invalid site_type values: {bad_types}")

    bad_groups = sorted(set(df["group_position"].unique()) - VALID_GROUPS)
    if bad_groups:
        raise ValueError(f"Invalid group_position values: {bad_groups}")

    if not set(df["label"].unique()).issubset({0, 1, 2}):
        raise ValueError("Label values must be subset of {0,1,2}.")

    return df


if INPUT_CSV_PATH.exists():
    raw_df = pd.read_csv(INPUT_CSV_PATH)
    df = validate_dataframe(raw_df)
    print("Loaded rows:", len(df))
    print("Label distribution:\n", df["label"].value_counts(normalize=True).round(4))
    print("Site type distribution:\n", df["site_type"].value_counts(normalize=True).round(4))
    print("Group distribution:\n", df["group_position"].value_counts(normalize=True).round(4))
else:
    df = pd.DataFrame(columns=REQUIRED_COLUMNS)
    print("Input CSV not found. Patch INPUT_CSV_PATH then rerun this cell.")

## 4) Tien xu ly chuoi va ma hoa dau vao

Cell nay tao tokenizer/encoding va DataLoader de suy luan batch.

In [ ]:
class SeqDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.df = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            "site_id": row["site_id"],
            "sequence": row["sequence"],
            "label": int(row["label"]),
            "site_type": row["site_type"],
            "group_position": row["group_position"],
        }


def collate_batch(batch):
    return {
        "site_id": [x["site_id"] for x in batch],
        "sequence": [x["sequence"] for x in batch],
        "label": torch.tensor([x["label"] for x in batch], dtype=torch.long),
        "site_type": [x["site_type"] for x in batch],
        "group_position": [x["group_position"] for x in batch],
    }


def get_offsets_or_none(tokenizer, sequences):
    try:
        tk = tokenizer(
            sequences,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024,
            return_offsets_mapping=True,
        )
        return tk
    except Exception:
        return tokenizer(
            sequences,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024,
        )


tokenizer = AutoTokenizer.from_pretrained(NT_MODEL_NAME)

if len(df) > 0:
    ds = SeqDataset(df)
    loader = DataLoader(ds, batch_size=8, shuffle=False, collate_fn=collate_batch)
    first_batch = next(iter(loader))
    print("Example batch size:", len(first_batch["sequence"]))
    print("Sequence length sample:", len(first_batch["sequence"][0]))
else:
    loader = None
    print("DataFrame is empty. Patch data path first.")

## 5) Nap mo hinh da huan luyen va cau hinh suy luan

Khoi tao composite model: NT encoder -> lay center embedding -> MLP classifier checkpoint.

In [ ]:
class SpliceSiteClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dims, dropout, num_classes):
        super().__init__()
        layers = []
        prev_dim = embedding_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, num_classes))
        self.classifier = nn.Sequential(*layers)

    def forward(self, embeddings):
        return self.classifier(embeddings)


class CompositeSpliceModel(nn.Module):
    def __init__(self, nt_model, head_model, center_index=300):
        super().__init__()
        self.nt_model = nt_model
        self.head_model = head_model
        self.center_index = center_index

    def forward(self, input_ids, attention_mask):
        out = self.nt_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_attentions=False,
            return_dict=True,
        )
        hidden = out.last_hidden_state  # [B, T, H]
        seq_len = attention_mask.sum(dim=1)  # [B]
        center_idx = torch.clamp(
            torch.full_like(seq_len, self.center_index),
            min=0,
            max=(hidden.size(1) - 1),
        )
        emb = hidden[torch.arange(hidden.size(0), device=hidden.device), center_idx]
        logits = self.head_model(emb)
        return logits


def infer_embedding_dim_from_state_dict(state_dict):
    # First linear weight shape: [hidden_dim, embedding_dim]
    for k, v in state_dict.items():
        if k.endswith("classifier.0.weight") and len(v.shape) == 2:
            return int(v.shape[1])
    # Fallback: detect first 2D tensor
    for _, v in state_dict.items():
        if len(v.shape) == 2:
            return int(v.shape[1])
    raise ValueError("Could not infer embedding_dim from checkpoint.")


nt_model = AutoModel.from_pretrained(NT_MODEL_NAME).to(DEVICE)
nt_model.eval()

head_model = None
composite_model = None

if MODEL_CHECKPOINT_PATH.exists():
    state = torch.load(MODEL_CHECKPOINT_PATH, map_location=DEVICE)

    if isinstance(state, dict):
        emb_dim = infer_embedding_dim_from_state_dict(state)
        head_model = SpliceSiteClassifier(
            embedding_dim=emb_dim,
            hidden_dims=HIDDEN_DIMS,
            dropout=DROPOUT,
            num_classes=NUM_CLASSES,
        ).to(DEVICE)
        head_model.load_state_dict(state, strict=False)
    else:
        # If a full module was saved
        head_model = state.to(DEVICE)

    head_model.eval()
    composite_model = CompositeSpliceModel(nt_model, head_model, center_index=CENTER_INDEX).to(DEVICE)
    composite_model.eval()
    print("Checkpoint loaded from:", MODEL_CHECKPOINT_PATH)
else:
    print("Checkpoint placeholder missing. Patch MODEL_CHECKPOINT_PATH then rerun.")


def predict_batch(sequences, model):
    tk = tokenizer(
        sequences,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024,
    )
    input_ids = tk["input_ids"].to(DEVICE)
    attention_mask = tk["attention_mask"].to(DEVICE)
    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(logits, dim=-1)
    return logits.detach().cpu().numpy(), probs.detach().cpu().numpy()

## 6) Chay du doan baseline va tinh metric

Tinh cac metric co ban va luu vao `outputs/metrics.csv`.

In [ ]:
def safe_multiclass_auc(y_true, probs):
    try:
        return roc_auc_score(y_true, probs, multi_class="ovr", average="macro")
    except Exception:
        return np.nan


def safe_multiclass_auprc(y_true, probs):
    try:
        y = pd.get_dummies(y_true).reindex(columns=[0, 1, 2], fill_value=0).values
        return average_precision_score(y, probs, average="macro")
    except Exception:
        return np.nan


if len(df) > 0 and composite_model is not None:
    logits, probs = predict_batch(df["sequence"].tolist(), composite_model)
    preds = probs.argmax(axis=1)
    y_true = df["label"].to_numpy()

    baseline_metrics = {
        "accuracy": float(accuracy_score(y_true, preds)),
        "f1_macro": float(f1_score(y_true, preds, average="macro")),
        "auroc_ovr_macro": float(safe_multiclass_auc(y_true, probs)),
        "auprc_macro": float(safe_multiclass_auprc(y_true, probs)),
    }

    metric_df = pd.DataFrame([baseline_metrics])
    metric_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
    print(metric_df)
else:
    baseline_metrics = {}
    probs = None
    print("Skip baseline inference. Ensure data + checkpoint are ready.")

## 7) Tinh toan interpretability (Integrated Gradients / Attention Rollout / ISM)

Cell nay tinh score dong gop theo vi tri nucleotide va luu ket qua attribution.

In [ ]:
def token_scores_to_nucleotide_scores(token_scores, offsets, seq_len=WINDOW_LEN):
    scores = np.zeros(seq_len, dtype=np.float32)
    counts = np.zeros(seq_len, dtype=np.float32)
    for s, (start, end) in zip(token_scores, offsets):
        if end <= start:
            continue
        start = max(0, min(seq_len, start))
        end = max(0, min(seq_len, end))
        if end <= start:
            continue
        scores[start:end] += float(s)
        counts[start:end] += 1.0
    counts[counts == 0] = 1.0
    return scores / counts


def get_tokenized_with_offsets(seq):
    try:
        tk = tokenizer(
            seq,
            return_tensors="pt",
            truncation=True,
            max_length=1024,
            return_offsets_mapping=True,
        )
        offsets = tk["offset_mapping"][0].cpu().numpy().tolist()
        tk.pop("offset_mapping")
        return tk, offsets
    except Exception:
        tk = tokenizer(seq, return_tensors="pt", truncation=True, max_length=1024)
        # Fallback: spread token contributions uniformly over sequence
        n_tok = tk["input_ids"].shape[1]
        step = max(1, WINDOW_LEN // max(1, n_tok))
        offsets = []
        cur = 0
        for _ in range(n_tok):
            nxt = min(WINDOW_LEN, cur + step)
            offsets.append([cur, nxt])
            cur = nxt
        if offsets:
            offsets[-1][1] = WINDOW_LEN
        return tk, offsets


def get_center_embedding_and_logits(seq, composite):
    tk = tokenizer(seq, return_tensors="pt", truncation=True, max_length=1024)
    input_ids = tk["input_ids"].to(DEVICE)
    attention_mask = tk["attention_mask"].to(DEVICE)

    with torch.no_grad():
        out = composite.nt_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_attentions=False,
            return_dict=True,
        )
        hidden = out.last_hidden_state
        idx = min(CENTER_INDEX, hidden.size(1) - 1)
        emb = hidden[:, idx, :]
        logits = composite.head_model(emb)
        probs = torch.softmax(logits, dim=-1)
    return emb, logits, probs


def compute_ig_for_sequence(seq, target_class, composite):
    if not CAPTUM_AVAILABLE:
        raise RuntimeError("Captum is not installed. Please install captum.")

    tk, offsets = get_tokenized_with_offsets(seq)
    input_ids = tk["input_ids"].to(DEVICE)
    attention_mask = tk["attention_mask"].to(DEVICE)

    # Use embedding-space IG on NT input embeddings
    input_embeds = composite.nt_model.embeddings(input_ids).detach()
    input_embeds.requires_grad_(True)

    baseline = torch.zeros_like(input_embeds)

    def forward_with_embeds(embeds):
        out = composite.nt_model(
            inputs_embeds=embeds,
            attention_mask=attention_mask,
            output_attentions=False,
            return_dict=True,
        )
        hidden = out.last_hidden_state
        idx = min(CENTER_INDEX, hidden.size(1) - 1)
        center_emb = hidden[:, idx, :]
        logits = composite.head_model(center_emb)
        return logits

    ig = IntegratedGradients(forward_with_embeds)
    attributions, _ = ig.attribute(
        inputs=input_embeds,
        baselines=baseline,
        target=int(target_class),
        return_convergence_delta=True,
    )

    token_scores = attributions.detach().abs().sum(dim=-1).squeeze(0).cpu().numpy()
    nt_scores = token_scores_to_nucleotide_scores(token_scores, offsets, seq_len=WINDOW_LEN)
    return nt_scores


def compute_attention_rollout_for_sequence(seq, composite):
    tk, offsets = get_tokenized_with_offsets(seq)
    input_ids = tk["input_ids"].to(DEVICE)
    attention_mask = tk["attention_mask"].to(DEVICE)

    with torch.no_grad():
        out = composite.nt_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_attentions=True,
            return_dict=True,
        )

    # attentions: tuple(num_layers) each [B, heads, T, T]
    attn_layers = [a[0].mean(dim=0).cpu().numpy() for a in out.attentions]  # [T, T]
    T = attn_layers[0].shape[0]
    rollout = np.eye(T, dtype=np.float32)

    for a in attn_layers:
        a = a + np.eye(T, dtype=np.float32)
        a = a / np.clip(a.sum(axis=-1, keepdims=True), a_min=1e-9, a_max=None)
        rollout = a @ rollout

    center_tok = min(CENTER_INDEX, T - 1)
    token_scores = rollout[center_tok]
    nt_scores = token_scores_to_nucleotide_scores(token_scores, offsets, seq_len=WINDOW_LEN)
    return nt_scores


def compute_ism_for_sequence(seq, target_class, composite):
    bases = ["A", "C", "G", "T"]

    _, logits, probs = get_center_embedding_and_logits(seq, composite)
    base_score = probs[0, int(target_class)].item()

    seq_list = list(seq)
    importance = np.zeros(WINDOW_LEN, dtype=np.float32)

    for i in range(WINDOW_LEN):
        orig = seq_list[i]
        if orig not in bases:
            importance[i] = 0.0
            continue

        deltas = []
        for b in bases:
            if b == orig:
                continue
            mut = seq_list.copy()
            mut[i] = b
            mut_seq = "".join(mut)
            _, _, mut_probs = get_center_embedding_and_logits(mut_seq, composite)
            mut_score = mut_probs[0, int(target_class)].item()
            deltas.append(base_score - mut_score)

        importance[i] = float(np.max(deltas)) if deltas else 0.0

    return importance


def choose_target_class(row):
    # If labels are available and trusted, use label.
    # Otherwise derive from site_type.
    if "label" in row and int(row["label"]) in {1, 2}:
        return int(row["label"])
    return 1 if row["site_type"] == "donor" else 2


def run_all_attributions(frame, composite, max_rows=None):
    work_df = frame.copy()
    if max_rows is not None:
        work_df = work_df.head(max_rows).copy()

    rows = []
    ig_all = []
    roll_all = []
    ism_all = []

    for _, row in work_df.iterrows():
        seq = row["sequence"]
        target_class = choose_target_class(row)

        ig_scores = compute_ig_for_sequence(seq, target_class, composite) if CAPTUM_AVAILABLE else np.zeros(WINDOW_LEN, dtype=np.float32)
        rollout_scores = compute_attention_rollout_for_sequence(seq, composite)
        ism_scores = compute_ism_for_sequence(seq, target_class, composite)

        ig_all.append(ig_scores)
        roll_all.append(rollout_scores)
        ism_all.append(ism_scores)

        rows.append({
            "site_id": row["site_id"],
            "label": int(row["label"]),
            "site_type": row["site_type"],
            "group_position": row["group_position"],
            "target_class": int(target_class),
        })

    meta = pd.DataFrame(rows)
    ig_arr = np.stack(ig_all) if ig_all else np.zeros((0, WINDOW_LEN), dtype=np.float32)
    roll_arr = np.stack(roll_all) if roll_all else np.zeros((0, WINDOW_LEN), dtype=np.float32)
    ism_arr = np.stack(ism_all) if ism_all else np.zeros((0, WINDOW_LEN), dtype=np.float32)

    np.save(ARTIFACTS_DIR / "ig_scores.npy", ig_arr)
    np.save(ARTIFACTS_DIR / "rollout_scores.npy", roll_arr)
    np.save(ARTIFACTS_DIR / "ism_scores.npy", ism_arr)
    meta.to_csv(OUTPUT_DIR / "attribution_meta.csv", index=False)

    return meta, ig_arr, roll_arr, ism_arr


if len(df) > 0 and composite_model is not None:
    # Set max_rows=None for full run
    meta_df, ig_scores, rollout_scores, ism_scores = run_all_attributions(df, composite_model, max_rows=None)
    print("Attribution done. Shapes:")
    print("IG:", ig_scores.shape, "Rollout:", rollout_scores.shape, "ISM:", ism_scores.shape)
else:
    meta_df = pd.DataFrame()
    ig_scores = np.zeros((0, WINDOW_LEN), dtype=np.float32)
    rollout_scores = np.zeros((0, WINDOW_LEN), dtype=np.float32)
    ism_scores = np.zeros((0, WINDOW_LEN), dtype=np.float32)
    print("Skip attributions. Ensure data + checkpoint are ready.")

## 8) Phan tich vung splice-site va motif quan trong

So sanh motif-only va motif-flank cho donor/acceptor, tong hop theo nhom vi tri.

In [ ]:
def motif_indices(site_type):
    # With your convention: center nucleotide is G of GT/AG.
    # donor motif core: G(center) + T(center+1)
    # acceptor motif core: A(center-1) + G(center)
    if site_type == "donor":
        return [CENTER_INDEX, CENTER_INDEX + 1]
    return [CENTER_INDEX - 1, CENTER_INDEX]


def motif_flank_indices(site_type, flank=10):
    core = motif_indices(site_type)
    left = max(0, min(core) - flank)
    right = min(WINDOW_LEN - 1, max(core) + flank)
    return list(range(left, right + 1))


def localization_at_k(scores, allowed_idx, k=10):
    if len(scores) == 0:
        return np.nan
    topk = np.argsort(-scores)[:k]
    allowed = set(allowed_idx)
    hit = np.isin(topk, list(allowed)).sum()
    return float(hit / max(1, k))


def junction_hit_rate(scores, core_idx, top_m=5):
    topm = set(np.argsort(-scores)[:top_m].tolist())
    return float(any(i in topm for i in core_idx))


def peak_distance(scores, core_idx):
    peak = int(np.argmax(scores))
    return float(min(abs(peak - i) for i in core_idx))


def motif_disruption_delta(scores, core_idx):
    return float(np.mean([scores[i] for i in core_idx]))


def evaluate_method(meta, method_scores, method_name, flank=10, k=10, top_m=5):
    rows = []
    for i, row in meta.iterrows():
        sc = method_scores[i]
        st = row["site_type"]
        grp = row["group_position"]
        core = motif_indices(st)
        flank_idx = motif_flank_indices(st, flank=flank)

        rows.append({
            "method": method_name,
            "site_id": row["site_id"],
            "site_type": st,
            "group_position": grp,
            "motif_only_junction_hit": junction_hit_rate(sc, core, top_m=top_m),
            "motif_only_peak_distance": peak_distance(sc, core),
            "motif_flank_localization_at_k": localization_at_k(sc, flank_idx, k=k),
            "motif_disruption_score": motif_disruption_delta(sc, core),
        })

    return pd.DataFrame(rows)


if len(meta_df) > 0:
    eval_ig = evaluate_method(meta_df, ig_scores, "IG", flank=10, k=10, top_m=5)
    eval_roll = evaluate_method(meta_df, rollout_scores, "Rollout", flank=10, k=10, top_m=5)
    eval_ism = evaluate_method(meta_df, ism_scores, "ISM", flank=10, k=10, top_m=5)

    eval_all = pd.concat([eval_ig, eval_roll, eval_ism], ignore_index=True)
    eval_all.to_csv(OUTPUT_DIR / "interpretability_site_level_metrics.csv", index=False)

    summary = (
        eval_all
        .groupby(["method", "site_type", "group_position"], as_index=False)
        .agg({
            "motif_only_junction_hit": "mean",
            "motif_only_peak_distance": "mean",
            "motif_flank_localization_at_k": "mean",
            "motif_disruption_score": "mean",
        })
        .sort_values(["method", "site_type", "group_position"])
    )
    summary.to_csv(OUTPUT_DIR / "interpretability_summary_by_group.csv", index=False)
    print(summary.head(20))
else:
    eval_all = pd.DataFrame()
    summary = pd.DataFrame()
    print("No attribution results to evaluate.")

## 9) Truc quan hoa ket qua va luu artifact

Ve heatmap attribution, profile trung binh theo vi tri va bieu do so sanh nhom.

In [ ]:
def plot_mean_profile(scores, title, save_path):
    if scores.shape[0] == 0:
        return
    mean_prof = scores.mean(axis=0)
    plt.figure(figsize=(14, 4))
    plt.plot(mean_prof, linewidth=1.2)
    plt.axvline(CENTER_INDEX, linestyle="--")
    plt.axvline(CENTER_INDEX - 1, linestyle="--", alpha=0.4)
    plt.axvline(CENTER_INDEX + 1, linestyle="--", alpha=0.4)
    plt.title(title)
    plt.xlabel("Nucleotide position in 601-window")
    plt.ylabel("Attribution score")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


def plot_heatmap(scores, title, save_path, max_rows=100):
    if scores.shape[0] == 0:
        return
    show = scores[:max_rows]
    plt.figure(figsize=(14, 6))
    sns.heatmap(show, cmap="viridis", cbar=True)
    plt.title(title)
    plt.xlabel("Nucleotide position")
    plt.ylabel("Samples")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()


if len(meta_df) > 0:
    plot_mean_profile(ig_scores, "IG mean profile", PLOTS_DIR / "ig_mean_profile.png")
    plot_mean_profile(rollout_scores, "Rollout mean profile", PLOTS_DIR / "rollout_mean_profile.png")
    plot_mean_profile(ism_scores, "ISM mean profile", PLOTS_DIR / "ism_mean_profile.png")

    plot_heatmap(ig_scores, "IG heatmap (top 100 rows)", PLOTS_DIR / "ig_heatmap.png")
    plot_heatmap(rollout_scores, "Rollout heatmap (top 100 rows)", PLOTS_DIR / "rollout_heatmap.png")
    plot_heatmap(ism_scores, "ISM heatmap (top 100 rows)", PLOTS_DIR / "ism_heatmap.png")

    # Method comparison by summary metrics
    if len(summary) > 0:
        for metric_col in [
            "motif_only_junction_hit",
            "motif_only_peak_distance",
            "motif_flank_localization_at_k",
            "motif_disruption_score",
        ]:
            plt.figure(figsize=(10, 4))
            sns.barplot(data=summary, x="method", y=metric_col, hue="site_type")
            plt.title(f"Method comparison - {metric_col}")
            plt.tight_layout()
            plt.savefig(PLOTS_DIR / f"compare_{metric_col}.png", dpi=150)
            plt.close()

    print("Plots saved to:", PLOTS_DIR)
else:
    print("No plots generated because attribution outputs are empty.")

## 10) Xuat bao cao tom tat va file ket qua

Cell cuoi tao bao cao markdown/csv tong hop va checklist tai lap end-to-end.

In [ ]:
from datetime import datetime

report = {
    "timestamp": datetime.now().isoformat(),
    "input_csv": str(INPUT_CSV_PATH),
    "checkpoint": str(MODEL_CHECKPOINT_PATH),
    "rows": int(len(df)) if "df" in globals() else 0,
    "window_len": WINDOW_LEN,
    "center_index": CENTER_INDEX,
    "methods": ["IG", "Rollout", "ISM"],
    "baseline_metrics": baseline_metrics,
    "artifacts": {
        "ig_npy": str(ARTIFACTS_DIR / "ig_scores.npy"),
        "rollout_npy": str(ARTIFACTS_DIR / "rollout_scores.npy"),
        "ism_npy": str(ARTIFACTS_DIR / "ism_scores.npy"),
        "site_level_csv": str(OUTPUT_DIR / "interpretability_site_level_metrics.csv"),
        "summary_csv": str(OUTPUT_DIR / "interpretability_summary_by_group.csv"),
        "plots_dir": str(PLOTS_DIR),
    },
}

with open(OUTPUT_DIR / "run_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

md_lines = [
    "# Interpretability Run Report",
    "",
    f"- Timestamp: {report['timestamp']}",
    f"- Input CSV: {report['input_csv']}",
    f"- Checkpoint: {report['checkpoint']}",
    f"- Rows: {report['rows']}",
    f"- Window length: {report['window_len']}",
    f"- Center index: {report['center_index']}",
    f"- Methods: {', '.join(report['methods'])}",
    "",
    "## Baseline Metrics",
]

if baseline_metrics:
    for k, v in baseline_metrics.items():
        md_lines.append(f"- {k}: {v:.6f}")
else:
    md_lines.append("- Baseline metrics unavailable (missing data/checkpoint).")

md_lines.extend([
    "",
    "## Output Artifacts",
])
for k, v in report["artifacts"].items():
    md_lines.append(f"- {k}: {v}")

with open(OUTPUT_DIR / "run_report.md", "w", encoding="utf-8") as f:
    f.write("\n".join(md_lines))

print("Report saved:")
print("-", OUTPUT_DIR / "run_report.json")
print("-", OUTPUT_DIR / "run_report.md")
print("\nNotebook end-to-end checklist:")
print("1) Patch INPUT_CSV_PATH + MODEL_CHECKPOINT_PATH in Section 1")
print("2) Run all cells from top to bottom")
print("3) Check outputs in outputs/, plots/, artifacts/")